<a href="https://colab.research.google.com/github/burrows99/melanoma-classification/blob/master/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Melanoma Classification — Colab Runner

Clones (or updates) the repo, sets up `uv`, configures Kaggle credentials, runs baseline + 4 experiment presets, then downloads plots and saved models.

In [ ]:
REPO_URL = "https://github.com/burrows99/melanoma-classification.git"
REPO_DIR = "melanoma-classification"

# Always start from /content to avoid nested clones
%cd /content

# Remove stale clone if present, then clone fresh
!rm -rf {REPO_DIR}
!git clone {REPO_URL}

%cd {REPO_DIR}
print("Working directory: ", end="")
!pwd

/content
Cloning into 'melanoma-classification'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 299 (delta 104), reused 128 (delta 57), pack-reused 113 (from 2)
Receiving objects: 100% (299/299), 11.73 MiB | 18.03 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/melanoma-classification
Working directory: /content/melanoma-classification


## Step 2 — Install uv and sync dependencies

In [ ]:
import os

# Install uv (idempotent — safe to run every time)
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this session
uv_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = uv_bin + os.pathsep + os.environ.get("PATH", "")

# Sync all dependencies from pyproject.toml
!uv sync
print("Dependencies synced.")

downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 123 packages in 1ms
Installed 112 packages in 302ms
 + albucore==0.0.24
 + albumentations==2.0.8
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + brotli==1.2.0
 + certifi==2026.2.25
 + charset-normalizer==3.4.7
 + click==8.3.2
 + cloudpickle==3.1.2
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.2
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + fastapi==0.135.3
 + filelock==3.25.2
 + fonttools==4.62.1
 + fsspec==2026.3.0
 + grad-cam==1.5.5
 + gradio==6.12.0
 + gradio-client==2.4.1
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.3.0
 + hf-xet==1.4.3
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.10.1
 + idna==3.11
 + jinja2==3.1.6
 + joblib==1.5.3
 + kagglehub==1.0.0
 + kagglesdk==0.1.18
 + kiwisolver==1.5.0
 + llvmlite==0.47.0

## Step 3 — Configure Kaggle credentials

Set your Kaggle username and API key below. You can find these at https://www.kaggle.com/settings → API → Create New Token.

In [ ]:
import os, json, pathlib

KAGGLE_USERNAME = "mrburrows"  # <-- replace
KAGGLE_KEY      = "KGAT_7ebf0cae77a8cd59de30d0e1a51f7dfc"   # <-- replace

# Write ~/.kaggle/kaggle.json (required by kagglehub)
kaggle_cfg = pathlib.Path.home() / ".kaggle" / "kaggle.json"
kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
kaggle_cfg.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
kaggle_cfg.chmod(0o600)

# Also export as environment variables (belt-and-suspenders)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

print("Kaggle credentials configured.")

Kaggle credentials configured.


## Step 4 — Run baseline + experiments

- **Baseline (A)** — Adam, no scheduler, focal γ=2.0
- **Experiment 1 (B)** — Adam + CosineAnnealing
- **Experiment 2 (C)** — AdamW + CosineAnnealing
- **Experiment 3 (D)** — AdamW + CosineAnnealing + focal γ=1.5
- **Experiment 4** — Image-only ablation (no metadata)

In [ ]:
import os
# Colab sets MPLBACKEND=module://matplotlib_inline.backend_inline which is
# inherited by uv run subprocesses but not installed in the venv — force Agg.
os.environ["MPLBACKEND"] = "Agg"
print("MPLBACKEND set to Agg")


MPLBACKEND set to Agg


### 4.1 — Baseline (A) — Adam, no scheduler, focal γ=2.0

In [ ]:
!uv run main.py --train

22:49:28  __main__  INFO  Dataset not found in CWD — downloading from Kaggle (cached after first run)...
Using Colab cache for faster access to the 'melanoma-external-malignant-256' dataset.
22:49:30  __main__  INFO  Dataset ready: 37648 images, CSV at dataset/train_concat.csv
Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
22:49:38  model.metadata_melanoma_model  INFO  Creating EfficientNet-B0 model with 14 metadata features.
22:49:39  train  INFO  Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:46<00:00,  8.81it/s]
Evaluating: 100% 236/236 [01:22<00:00,  2.87it/s]
22:52:48  train  INFO  Train: Loss=0.0152 | Acc=0.9010 | Recall=0.8879 | F1=0.7088
22:52:48  train  INFO  Val:   Loss=0.0099 | Acc=0.9521 | Recall=0.9207 | F1=0.8389
22:52:48  train  INFO  New best model saved: output/efficientnet_b0/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep20_BasicAug_best_ep1.pth  (Val F1: 0.8389)
22:52:48  train  INFO  Epoch 2/20  |  LR: 0.0

### 4.2 — Experiment 1 (B) — Adam + CosineAnnealing

In [ ]:
!uv run main.py --train --experiment 1

Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
23:53:21  model.metadata_melanoma_model  INFO  Creating EfficientNet-B0 model with 14 metadata features.
23:53:21  train  INFO  Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:44<00:00,  9.03it/s]
Evaluating: 100% 236/236 [01:20<00:00,  2.94it/s]
23:56:26  train  INFO  Train: Loss=0.0159 | Acc=0.8946 | Recall=0.8888 | F1=0.6956
23:56:26  train  INFO  Val:   Loss=0.0104 | Acc=0.9356 | Recall=0.9265 | F1=0.7960
23:56:26  train  INFO  New best model saved: output/experiment1/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep20_BasicAug_best_ep1.pth  (Val F1: 0.7960)
23:56:26  train  INFO  Epoch 2/20  |  LR: 9.938441702975689e-05
Training: 100% 941/941 [01:43<00:00,  9.09it/s]
Evaluating: 100% 236/236 [01:19<00:00,  2.96it/s]
23:59:29  train  INFO  Train: Loss=0.0122 | Acc=0.9287 | Recall=0.9140 | F1=0.7766
23:59:29  train  INFO  Val:   Loss=0.0099 | Acc=0.9582 | Recall=0.9158 | F1=0.8558

### 4.3 — Experiment 2 (C) — AdamW + CosineAnnealing

In [ ]:
!uv run main.py --train --experiment 2

Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
00:56:16  model.metadata_melanoma_model  INFO  Creating EfficientNet-B0 model with 14 metadata features.
00:56:16  train  INFO  Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:46<00:00,  8.83it/s]
Evaluating: 100% 236/236 [01:21<00:00,  2.90it/s]
00:59:24  train  INFO  Train: Loss=0.0155 | Acc=0.9002 | Recall=0.8881 | F1=0.7070
00:59:24  train  INFO  Val:   Loss=0.0114 | Acc=0.9507 | Recall=0.9138 | F1=0.8342
00:59:24  train  INFO  New best model saved: output/experiment2/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep20_BasicAug_best_ep1.pth  (Val F1: 0.8342)
00:59:24  train  INFO  Epoch 2/20  |  LR: 9.938441702975689e-05
Training: 100% 941/941 [01:42<00:00,  9.15it/s]
Evaluating: 100% 236/236 [01:19<00:00,  2.97it/s]
01:02:27  train  INFO  Train: Loss=0.0119 | Acc=0.9301 | Recall=0.9114 | F1=0.7796
01:02:27  train  INFO  Val:   Loss=0.0099 | Acc=0.9401 | Recall=0.9373 | F1=0.8093

### 4.4 — Experiment 3 (D) — AdamW + CosineAnnealing + focal γ=1.5

In [ ]:
!uv run main.py --train --experiment 3

Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
02:00:19  model.metadata_melanoma_model  INFO  Creating EfficientNet-B0 model with 14 metadata features.
02:00:19  train  INFO  Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:43<00:00,  9.10it/s]
Evaluating: 100% 236/236 [01:16<00:00,  3.09it/s]
02:03:19  train  INFO  Train: Loss=0.0214 | Acc=0.8989 | Recall=0.8942 | F1=0.7057
02:03:19  train  INFO  Val:   Loss=0.0144 | Acc=0.9429 | Recall=0.9275 | F1=0.8150
02:03:19  train  INFO  New best model saved: output/experiment3/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep20_BasicAug_best_ep1.pth  (Val F1: 0.8150)
02:03:19  train  INFO  Epoch 2/20  |  LR: 9.938441702975689e-05
Training: 100% 941/941 [01:43<00:00,  9.12it/s]
Evaluating: 100% 236/236 [01:20<00:00,  2.95it/s]
02:06:23  train  INFO  Train: Loss=0.0164 | Acc=0.9325 | Recall=0.9111 | F1=0.7854
02:06:23  train  INFO  Val:   Loss=0.0136 | Acc=0.9450 | Recall=0.9344 | F1=0.8217

### 4.5 — Experiment 4 — Image-only ablation (no metadata)

In [ ]:
!uv run main.py --train --experiment 4

Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
03:03:13  model.metadata_melanoma_model  INFO  Creating EfficientNet-B0 model with 14 metadata features.
03:03:13  train  INFO  Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:43<00:00,  9.07it/s]
Evaluating: 100% 236/236 [01:19<00:00,  2.97it/s]
03:06:16  train  INFO  Train: Loss=0.0168 | Acc=0.8883 | Recall=0.8827 | F1=0.6819
03:06:16  train  INFO  Val:   Loss=0.0113 | Acc=0.9506 | Recall=0.8903 | F1=0.8301
03:06:16  train  INFO  New best model saved: output/experiment4/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep20_BasicAug_best_ep1.pth  (Val F1: 0.8301)
03:06:16  train  INFO  Epoch 2/20  |  LR: 0.0001
Training: 100% 941/941 [01:44<00:00,  9.04it/s]
Evaluating: 100% 236/236 [01:19<00:00,  2.95it/s]
03:09:21  train  INFO  Train: Loss=0.0131 | Acc=0.9270 | Recall=0.8979 | F1=0.7694
03:09:21  train  INFO  Val:   Loss=0.0104 | Acc=0.9579 | Recall=0.9070 | F1=0.8538
03:09:21  trai